# Credits

## Import and read file

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import ast
import csv, io

In [2]:
root = '../data'

In [3]:
credits = pd.read_csv(root + '/credits.csv', converters={'cast': ast.literal_eval, 'crew': ast.literal_eval}) # converters were added after inspection

print(f"There are {credits.shape[0]} rows and {credits.shape[1]} columns in the credits dataset")

display(credits.head())

There are 45476 rows and 3 columns in the credits dataset


,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': 'Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


## cast

In [4]:
def explode(df, column_names):
    exploded = df.explode().dropna()           
    df_exploded = pd.DataFrame()
    for column_name in column_names:
        df_exploded[column_name] = exploded.apply(lambda x: x[column_name])
    return df_exploded

# find keys
keys = credits.cast.iloc[0][0].keys()
exploded_cast = explode(credits.cast, keys)

# check unique ids
exploded_cast.id.describe()
print(f"There are {len(exploded_cast.id.unique())} ({len(exploded_cast.id.unique())/len(exploded_cast.id):.1%}) unique ids")

# it appears that id is per actor
# display(exploded_cast[exploded_cast['name'] == 'Tom Hanks'])

# ids that map to more than one name
names_per_id = exploded_cast.groupby('id')['name'].nunique()
dup_ids = names_per_id[names_per_id > 1].index
print(f"Nr of ids with multiple names: {len(dup_ids)}")
display(exploded_cast.loc[exploded_cast.id.isin(dup_ids), ['id', 'name']].drop_duplicates().sort_values(['id', 'name']))

# names that map to more than one id
ids_per_name = exploded_cast.groupby(['name'])['id'].nunique()
dup_names = ids_per_name[ids_per_name > 1].index
print(f"Nr of names with multiple ids: {len(dup_names)}")
display(exploded_cast.loc[exploded_cast.name.isin(dup_names), ['name', 'id']].drop_duplicates().sort_values(['name', 'id']).head(6))


There are 206158 (36.7%) unique ids
Nr of ids with multiple names: 14


,id,name
42005,9779,Morris Chestnut
201,9779,Моррис Честнат
45473,23764,Erika Eleniak
291,23764,Эрика Элениак
45473,58646,Damian Chapa
388,58646,Дэмиан Чапа
42225,67212,Tom Wu
6018,67212,Том Ву
39559,72932,Ryuhei Matsuda
8076,72932,松田龙平


Nr of names with multiple ids: 3053


,name,id
38225,Aaron Harrison,459878
15977,Aaron Harrison,1200414
6857,Aaron Jackson,20633
20570,Aaron Jackson,1435850
24342,Aaron Jackson,1780265
6352,Aaron Jeffery,92172


In [ ]:
# inspect gender should be mapped to something else than gender
display(exploded_cast.gender.value_counts())
display(exploded_cast)

gender
2    226713
0    223964
1    111797
Name: count, dtype: int64

,cast_id,character,credit_id,gender,id,name,order,profile_path
0,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg
0,15,Buzz Lightyear (voice),52fe4284c3a36847f8024f99,2,12898,Tim Allen,1,/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg
0,16,Mr. Potato Head (voice),52fe4284c3a36847f8024f9d,2,7167,Don Rickles,2,/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg
0,17,Slinky Dog (voice),52fe4284c3a36847f8024fa1,2,12899,Jim Varney,3,/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg
0,18,Rex (voice),52fe4284c3a36847f8024fa5,2,12900,Wallace Shawn,4,/oGE6JqPP2xH4tNORKNqxbNPYi7u.jpg
...,...,...,...,...,...,...,...,...
45474,2,,52fe4ea59251416c7515d7d5,2,544742,Iwan Mosschuchin,0,None
45474,3,,52fe4ea59251416c7515d7d9,1,1090923,Nathalie Lissenko,1,None
45474,4,,52fe4ea59251416c7515d7dd,2,1136422,Pavel Pavlov,2,None
45474,5,,52fe4ea59251416c7515d7e1,0,1261758,Aleksandr Chabrov,3,None


## crew

In [ ]:
def explode(df, column_names):
    exploded = df.explode().dropna()           
    df_exploded = pd.DataFrame()
    for column_name in column_names:
        df_exploded[column_name] = exploded.apply(lambda x: x[column_name])
    return df_exploded

# find keys
keys = credits.crew.iloc[0][0].keys()
display(keys)

exploded_crew = explode(credits.crew, keys)
display(exploded_crew.head())

exploded_crew.describe()

dict_keys(['credit_id', 'department', 'gender', 'id', 'job', 'name', 'profile_path'])

,credit_id,department,gender,id,job,name,profile_path
0,52fe4284c3a36847f8024f49,Directing,2,7879,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
0,52fe4284c3a36847f8024f4f,Writing,2,12891,Screenplay,Joss Whedon,/dTiVsuaTVTeGmvkhcyJvKp2A5kr.jpg
0,52fe4284c3a36847f8024f55,Writing,2,7,Screenplay,Andrew Stanton,/pvQWsu0qc8JFQhMVJkTHuexUAa1.jpg
0,52fe4284c3a36847f8024f5b,Writing,2,12892,Screenplay,Joel Cohen,/dAubAiZcvKFbboWlj7oXOkZnTSu.jpg
0,52fe4284c3a36847f8024f61,Writing,0,12893,Screenplay,Alec Sokolow,/v79vlRYi94BZUQnkkyznbGUZLjT.jpg


,gender,id
count,464314.000000,4.643140e+05
mean,0.759975,6.456037e+05
std,0.935607,6.791434e+05
min,0.000000,1.000000e+00
25%,0.000000,3.025600e+04
50%,0.000000,1.347900e+05
75%,2.000000,1.388851e+06
max,2.000000,1.908001e+06


# id

In [ ]:
movie_ids = movies.id.unique()

There appears to be 14 ids associated with multiple names, when inspecting the names it appears to be the same name in a different language. 

There are 3053 names associated with different ids